In [1]:
from pathlib import Path

base_dir = Path.cwd().parent
print(base_dir)

c:\Users\kirta\Desktop\IITD


In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import binomtest

def accuracy(df):
    return (df['parsed_answer'].str.lower() == df['answer'].str.lower()).mean()

def mean_confidence_overall(df):
    return df['confidence'].mean()

def confidence_accuracy_gap(df):   
    return df['confidence'].mean()/100 - accuracy(df)

def mean_confidence_correct(df):
    return df[df['parsed_answer'] == df['answer']]['confidence'].mean()

def mean_confidence_incorrect(df):
    return df[df['parsed_answer'] != df['answer']]['confidence'].mean()

def mean_confidence_change(df1, df2):
    return df2['confidence'].mean() - df1['confidence'].mean()

def brier(df):
    c = df['confidence']/100
    y = (df['parsed_answer'] == df['answer']).astype(float)
    return ((c - y)**2).mean()

def answer_flip_overall(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return (df_merged['parsed_answer_1'] != df_merged['parsed_answer_2']).sum()

def answer_flip_correct_to_incorrect(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return ((df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])).sum()

def confidence_change_correct_to_incorrect(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return df_merged[(df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])]['confidence_2'].mean() - df_merged[(df_merged['parsed_answer_1'] == df_merged['answer_1']) & (df_merged['parsed_answer_2'] != df_merged['answer_2'])]['confidence_1'].mean()


def answer_flip_incorrect_to_correct(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return ((df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])).sum()

def confidence_change_incorrect_to_correct(df1, df2):
    df_merged = df1.merge(
        df2, 
        on='id', 
        suffixes=('_1', '_2')
    )
    return df_merged[(df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])]['confidence_2'].mean() - df_merged[(df_merged['parsed_answer_1'] != df_merged['answer_1']) & (df_merged['parsed_answer_2'] == df_merged['answer_2'])]['confidence_1'].mean()

def compare_evidence(df1, df2):
    df_merged = df1.merge(
        df2,
        on="id",
        suffixes=("_1", "_2")
    )

    evidence1 = df_merged["evidence_1"].astype(str).tolist()
    evidence2 = df_merged["evidence_2"].astype(str).tolist()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )

    n = len(evidence1)
    vectors = vectorizer.fit_transform(evidence1 + evidence2)

    df_merged["evidence_similarity"] = cosine_similarity(
        vectors[:n], vectors[n:]
    ).diagonal()

    return df_merged.sort_values(
        "evidence_similarity",
        ascending=True
    )

In [3]:
import pandas as pd
df = pd.read_csv(f'{base_dir}/meta_csv/4_baseline_meta_results.csv')

In [4]:
pd.set_option('display.max_colwidth', None)

print('ACCURACY (CLEAN)', accuracy(df))

print('MEAN CONFIDENCE OVERALL (CLEAN)', mean_confidence_overall(df))

print('MEAN CONFIDENCE WHEN CORRECT (CLEAN)', mean_confidence_correct(df))

print('MEAN CONFIDENCE WHEN INCORRECT (CLEAN)', mean_confidence_incorrect(df))

print('CONFIDENCE-ACCURACY GAP, mean conf/100 - accuracy (CLEAN)', confidence_accuracy_gap(df))

print('BRIER SCORE (CLEAN)', brier(df))

ACCURACY (CLEAN) 0.415
MEAN CONFIDENCE OVERALL (CLEAN) 87.95
MEAN CONFIDENCE WHEN CORRECT (CLEAN) 87.65060240963855
MEAN CONFIDENCE WHEN INCORRECT (CLEAN) 88.16239316239316
CONFIDENCE-ACCURACY GAP, mean conf/100 - accuracy (CLEAN) 0.4645000000000001
BRIER SCORE (CLEAN) 0.463825


In [5]:
from sklearn.metrics import roc_auc_score

df["is_correct"] = (df["answer"].astype(str).str.strip().str.lower() == df["parsed_answer"].astype(str).str.strip().str.lower()).astype(int)

auroc = roc_auc_score(df["is_correct"], df["confidence"])

print("AUROC:", auroc)

AUROC: 0.4834208629389352


In [6]:
import pandas as pd
df = pd.read_csv(f'{base_dir}/qwen_csv/4_baseline_qwen_results.csv')

In [7]:
pd.set_option('display.max_colwidth', None)

print('ACCURACY (CLEAN)', accuracy(df))

print('MEAN CONFIDENCE OVERALL (CLEAN)', mean_confidence_overall(df))

print('MEAN CONFIDENCE WHEN CORRECT (CLEAN)', mean_confidence_correct(df))

print('MEAN CONFIDENCE WHEN INCORRECT (CLEAN)', mean_confidence_incorrect(df))

print('CONFIDENCE-ACCURACY GAP, mean conf/100 - accuracy (CLEAN)', confidence_accuracy_gap(df))

print('BRIER SCORE (CLEAN)', brier(df))

ACCURACY (CLEAN) 0.315
MEAN CONFIDENCE OVERALL (CLEAN) 94.375
MEAN CONFIDENCE WHEN CORRECT (CLEAN) 94.92063492063492
MEAN CONFIDENCE WHEN INCORRECT (CLEAN) 94.12408759124088
CONFIDENCE-ACCURACY GAP, mean conf/100 - accuracy (CLEAN) 0.6287499999999999
BRIER SCORE (CLEAN) 0.6082625


In [8]:
from sklearn.metrics import roc_auc_score

df["is_correct"] = (df["answer"].astype(str).str.strip().str.lower() == df["parsed_answer"].astype(str).str.strip().str.lower()).astype(int)

auroc = roc_auc_score(df["is_correct"], df["confidence"])

print("AUROC:", auroc)

AUROC: 0.5670258370988298


In [9]:
df.columns

Index(['id', 'source_subset', 'question', 'answer', 'options', 'injections',
       'type1_injection_key', 'injection_content', 'injection_provenance',
       'mh_terms', 'row_index', 'prompt', 'raw_model_output', 'parsed_answer',
       'confidence', 'evidence', 'parse_status', 'status', 'model_name',
       'model_revision', 'max_new_tokens', 'do_sample', 'is_correct'],
      dtype='object')

In [10]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

df_llama = pd.read_csv(f'{base_dir}/meta_csv/4_baseline_meta_results.csv')
df_qwen = pd.read_csv(f'{base_dir}/qwen_csv/4_baseline_qwen_results.csv')

df = df_llama[["id", "answer", "parsed_answer"]].merge(df_qwen[["id", "parsed_answer"]],on="id",suffixes=("_llama", "_qwen"))

# Determine whether each model is correct
df["llama_correct"] = (df["answer"] == df["parsed_answer_llama"])

df["qwen_correct"] = (df["answer"] == df["parsed_answer_qwen"])


llama_right_qwen_wrong = (df["llama_correct"] & ~df["qwen_correct"]).sum()

qwen_right_llama_wrong = (~df["llama_correct"] & df["qwen_correct"]).sum()

table = [
    [
        (df["llama_correct"] & df["qwen_correct"]).sum(),
        llama_right_qwen_wrong
    ],
    [
        qwen_right_llama_wrong,
        (~df["llama_correct"] & ~df["qwen_correct"]).sum()
    ]
]

result = mcnemar(table, exact=True)

print("Number of paired items:", len(df))
print("Llama correct, Qwen wrong:", llama_right_qwen_wrong)
print("Qwen correct, Llama wrong:", qwen_right_llama_wrong)
print("Paired McNemar p-value:", result.pvalue)

Number of paired items: 200
Llama correct, Qwen wrong: 41
Qwen correct, Llama wrong: 21
Paired McNemar p-value: 0.015134080997447081
